# Praat Scripts (日本語版) - フォルマントデータの一括抽出 (Google Colab版) 🎙️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okawawaka/praat-scripts-ja/blob/main/notebooks/collect_formant_data.ipynb)

本ノートブックは、`scripts/acoustic-analysis/collect_formant_data.praat` と**全く同一のアルゴリズム（Burg法 ＆ 線形補間）**で、母音区間の中央点におけるフォルマント周波数（F1〜F5）および帯域幅（B1〜B3）を一括抽出するツールです。

Praat の C++ エンジンそのものを内蔵した `praat-parselmouth` を使用しているため、**Praat本体で測定した場合と100%完全に同一の数値**が得られます。

### Step 1: 必要なライブラリのインストール

In [ ]:
!pip install -q praat-parselmouth textgrid pandas

### Step 2: 分析パラメータの設定
話者の声の高さに合わせて最大フォルマント周波数（目安：女性 5500 Hz、男性 5000 Hz、子供 8000 Hz）を設定してください。

In [ ]:
# @title 【パラメータ設定】 { run: "auto" }
target_tier = 1 # @param {type:"integer"}
max_formant_hz = 5500 # @param {type:"number"}
max_num_formants = 5 # @param {type:"integer"}
skip_empty_labels = False # @param {type:"boolean"}

print(f"設定完了: Tier={target_tier}, 最大フォルマント={max_formant_hz} Hz, 最大数={max_num_formants}")

### Step 3: 音声ファイル(.wav)とTextGrid(.TextGrid)をアップロード
下のセルを実行して表示される「ファイル選択」ボタンから、**分析したいペア（.wav と .TextGrid）をまとめて選択**してください（複数ペア可）。

In [ ]:
import os
import tempfile
import parselmouth
from parselmouth.praat import call
import textgrid
import pandas as pd
from google.colab import files

print("▼ 下の『ファイル選択』ボタンをクリックして、.wav と .TextGrid ファイルをまとめて選択してください：")
uploaded = files.upload()

if not uploaded:
    print("ファイルが選択されませんでした。")
else:
    # 一時ディレクトリに全ファイルを書き出し
    temp_dir = tempfile.mkdtemp()
    for fname, data in uploaded.items():
        with open(os.path.join(temp_dir, fname), "wb") as f:
            f.write(data)

    # wavファイルを基準に対応するTextGridを探索
    all_rows = []
    wav_files = [f for f in uploaded.keys() if f.lower().endswith(".wav")]

    for w_name in wav_files:
        base = os.path.splitext(w_name)[0]
        tg_candidates = [base + ".TextGrid", base + ".textgrid"]
        tg_name = next((c for c in tg_candidates if c in uploaded), None)

        if not tg_name:
            print(f"スキップ: {w_name} に対応する TextGrid が見つかりません。")
            continue

        wav_path = os.path.join(temp_dir, w_name)
        tg_path = os.path.join(temp_dir, tg_name)

        try:
            sound = parselmouth.Sound(wav_path)
            # Praat公式のBurg法でフォルマント解析オブジェクトを生成
            formant = sound.to_formant_burg(
                time_step=0.01,
                max_number_of_formants=max_num_formants,
                maximum_formant=max_formant_hz,
                window_length=0.025,
                pre_emphasis_from=50.0
            )

            tg = textgrid.TextGrid.fromFile(tg_path)
            if target_tier <= len(tg.tiers):
                tier = tg.tiers[target_tier - 1]
                if isinstance(tier, textgrid.IntervalTier):
                    for idx, interval in enumerate(tier, start=1):
                        label = interval.mark.strip()
                        if skip_empty_labels and label == "":
                            continue

                        start_t = interval.minTime
                        end_t = interval.maxTime
                        duration_ms = (end_t - start_t) * 1000.0
                        mid_t = start_t + ((end_t - start_t) / 2.0)

                        # Praatスクリプトと全く同一の線形補間（Linear）でフォルマント値と帯域幅を取得
                        f_vals = {}
                        for f_i in range(1, 6):
                            val = call(formant, "Get value at time", f_i, mid_t, "Hertz", "Linear")
                            f_vals[f"F{f_i}_Hz"] = f"{val:.2f}" if val is not None and not pd.isna(val) else "NA"

                        b_vals = {}
                        for b_i in range(1, 4):
                            b_val = call(formant, "Get bandwidth at time", b_i, mid_t, "Hertz", "Linear")
                            b_vals[f"B{b_i}_Hz"] = f"{b_val:.2f}" if b_val is not None and not pd.isna(b_val) else "NA"

                        row = {
                            "Filename": base,
                            "IntervalIndex": idx,
                            "Label": label,
                            "StartTime_s": f"{start_t:.4f}",
                            "EndTime_s": f"{end_t:.4f}",
                            "Duration_ms": f"{duration_ms:.2f}",
                            "MidTime_s": f"{mid_t:.4f}",
                            **f_vals,
                            **b_vals
                        }
                        all_rows.append(row)
        except Exception as e:
            print(f"エラー ({base}): {e}")

    if all_rows:
        df_formants = pd.DataFrame(all_rows)
        print(f"\n集計完了: 合計 {len(df_formants)} 件のフォルマントデータを抽出しました。")
    else:
        df_formants = pd.DataFrame()
        print("\n集計対象のデータがありませんでした。")

### Step 4: 抽出結果の確認（Praatと完全同一のテーブル形式）

In [ ]:
if 'df_formants' in locals() and not df_formants.empty:
    display(df_formants)
else:
    print("表示するデータがありません。")

### Step 5: TSVファイルのダウンロード

In [ ]:
if 'df_formants' in locals() and not df_formants.empty:
    out_tsv = "formant_results.tsv"
    df_formants.to_csv(out_tsv, sep="\t", index=False, encoding="utf-8-sig")
    print(f"{out_tsv} をダウンロードします...")
    files.download(out_tsv)
else:
    print("ダウンロード対象のデータがありません。")